In [1]:
import pandas as pd
import requests

In [2]:
url2015 = 'https://servicios.ine.es/wstempus/js/es/DATOS_TABLA/31097?tip=A&tv=847:&date=20150101:20150101'
url2016 = 'https://servicios.ine.es/wstempus/js/es/DATOS_TABLA/31097?tip=A&tv=847:&date=20160101:20160101'
url2017 = 'https://servicios.ine.es/wstempus/js/es/DATOS_TABLA/31097?tip=A&tv=847:&date=20170101:20170101'
url2018 = 'https://servicios.ine.es/wstempus/js/es/DATOS_TABLA/31097?tip=A&tv=847:&date=20180101:20180101'
url2019 = 'https://servicios.ine.es/wstempus/js/es/DATOS_TABLA/31097?tip=A&tv=847:&date=20190101:20190101'
url2020 = 'https://servicios.ine.es/wstempus/js/es/DATOS_TABLA/31097?tip=A&tv=847:&date=20200101:20200101'
url2021 = 'https://servicios.ine.es/wstempus/js/es/DATOS_TABLA/31097?tip=A&tv=847:&date=20210101:20210101'
url2022 = 'https://servicios.ine.es/wstempus/js/es/DATOS_TABLA/31097?tip=A&tv=847:&date=20220101:20220101'

In [3]:
def procesar_datos_ine(url2016):
    api_sec = requests.get(url2016)
    datos_json = api_sec.json()
    df = pd.DataFrame(datos_json)

    df_expandido = df.explode("Data").copy()

    df_expandido["Año"] = df_expandido["Data"].str["Anyo"].astype("Int64")
    df_expandido["Valor"] = df_expandido["Data"].str["Valor"].astype(float)

    filtro_madrid = df_expandido["Nombre"].str.contains("^Madrid", case=False, na=False)
    filtro_T3unidad = df_expandido["T3_Unidad"].str.strip() == "Euros"
    df_sec_filtrado = df_expandido[filtro_madrid & filtro_T3unidad].copy()

    separacion_nombre = df_sec_filtrado["Nombre"].str.split(".", n=1, expand=True)
    df_sec_filtrado["Seccion madrid"] = separacion_nombre[0].str.strip()
    df_sec_filtrado["Dato base"] = separacion_nombre[1].str.strip()

    indicadores_datobase = ["Dato base. Renta neta media por persona.", "Dato base. Renta neta media por hogar.",]
    filtro_rentas_sec = df_sec_filtrado["Dato base"].isin(indicadores_datobase)
    df_sec_filtrado = df_sec_filtrado[filtro_rentas_sec]
    df_sec_filtrado["Seccion censal"] = df_sec_filtrado["Seccion madrid"].str[-5:]
    

    columnas_drop1 = ["Data", "T3_Escala", "T3_Unidad", "Nombre", "Seccion madrid"]
    df_madrid_seccion = df_sec_filtrado.drop(columns=columnas_drop1).rename(columns={"Fecha": "Año"})

    return df_madrid_seccion

In [4]:
df_madrid_seccion_2015 = procesar_datos_ine(url2015)
df_madrid_seccion_2016 = procesar_datos_ine(url2016)
df_madrid_seccion_2017 = procesar_datos_ine(url2017)
df_madrid_seccion_2018 = procesar_datos_ine(url2018)
df_madrid_seccion_2019 = procesar_datos_ine(url2019)
df_madrid_seccion_2020 = procesar_datos_ine(url2020)
df_madrid_seccion_2021 = procesar_datos_ine(url2021)
df_madrid_seccion_2022 = procesar_datos_ine(url2022)

In [5]:
tablas = [df_madrid_seccion_2015, df_madrid_seccion_2016, df_madrid_seccion_2017, df_madrid_seccion_2018, df_madrid_seccion_2019, df_madrid_seccion_2020, df_madrid_seccion_2021, df_madrid_seccion_2022]
df_madrid_seccion_completo = pd.concat(tablas, axis=0, ignore_index=True)

In [6]:
df_madrid_seccion_completo

,COD,Año,Valor,Dato base,Seccion censal
0,ADRH20975,2015,22948.0,Dato base. Renta neta media por persona.,01001
1,ADRH20974,2015,49363.0,Dato base. Renta neta media por hogar.,01001
2,ADRH20973,2015,15045.0,Dato base. Renta neta media por persona.,01002
3,ADRH20972,2015,29040.0,Dato base. Renta neta media por hogar.,01002
4,ADRH20971,2015,15177.0,Dato base. Renta neta media por persona.,01003
...,...,...,...,...,...
39883,ADRH40872,2022,69852.0,Dato base. Renta neta media por hogar.,21032
39884,ADRH7226132,2022,14174.0,Dato base. Renta neta media por persona.,21033
39885,ADRH7226131,2022,34040.0,Dato base. Renta neta media por hogar.,21033
39886,ADRH10218871,2022,16796.0,Dato base. Renta neta media por persona.,21034


In [7]:
df_madrid_seccion_completo['Año'].isna().sum()

np.int64(876)

In [8]:
df_madrid_seccion_completo["tipo"] = df_madrid_seccion_completo["Dato base"].str.contains("hogar").map({True: "hogar", False: "persona"})
df_madrid_seccion_completo = df_madrid_seccion_completo.drop(columns=['COD', 'Dato base'])
df_madrid_seccion_completo = df_madrid_seccion_completo.rename(columns={'Seccion censal': 'COD_SEC_MADRID'})
df_madrid_seccion_completo['COD_SEC_MADRID'] = "28079" + df_madrid_seccion_completo['COD_SEC_MADRID'].astype(str)
df_madrid_seccion_completo


,Año,Valor,COD_SEC_MADRID,tipo
0,2015,22948.0,2807901001,persona
1,2015,49363.0,2807901001,hogar
2,2015,15045.0,2807901002,persona
3,2015,29040.0,2807901002,hogar
4,2015,15177.0,2807901003,persona
...,...,...,...,...
39883,2022,69852.0,2807921032,hogar
39884,2022,14174.0,2807921033,persona
39885,2022,34040.0,2807921033,hogar
39886,2022,16796.0,2807921034,persona


In [9]:
df = df_madrid_seccion_completo.reset_index(drop=True).copy()
df["año_pos"] = 2015 + df.index // 4986

In [10]:
df["Año"] = df["Año"].fillna(df["año_pos"]).astype(int)
df = df.drop(columns="año_pos")

In [ ]:
renta_sec_madrid = (df.pivot(index=["COD_SEC_MADRID", "Año"], columns="tipo", values="Valor")
           .reset_index().rename(columns={"hogar": "renta_hogar", "persona": "renta_persona"}))
renta_sec_madrid.columns.name = None  # ESTO ES PARA BORRAR EL TITULO 'TIPO' QUE SE GENERA POR EL PIVOT
print(renta_sec_madrid.shape) 
renta_sec_madrid

(19944, 4)


,COD_SEC_MADRID,Año,renta_hogar,renta_persona
0,2807901001,2015,49363.0,22948.0
1,2807901001,2016,48511.0,22891.0
2,2807901001,2017,48783.0,23122.0
3,2807901001,2018,58594.0,27785.0
4,2807901001,2019,66667.0,31119.0
...,...,...,...,...
19939,2807921034,2018,NaN,NaN
19940,2807921034,2019,NaN,NaN
19941,2807921034,2020,NaN,NaN
19942,2807921034,2021,NaN,NaN


In [12]:
#aqui la renta seccion ya estan limpios para cruce